# Spacy Custom NER: Stack Exchange Posts

> **Note**: This notebook uses `data/Posts.xml` from the provided dataset. 
> Analysis showed this specific file contains **Meta Math** discussions (e.g., about site features, moderation) rather than pure Math/Science questions. 
> However, the pipeline below is designed to be **generic**: it treats the Post's `Tags` as valid "TOPIC" entities. 
> 
> **To use for Math/Science**: Simply replace `data/Posts.xml` with the `Posts.xml` from the main **Math Stack Exchange** or **Science Stack Exchange** dump, and this code will automatically learn meaningful topics like "algebra", "calculus", "thermodynamics", etc.


In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans
from tqdm import tqdm
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import random
import os

In [ ]:
# Download the larger English model to use as a base or for comparison
# !python -m spacy download en_core_web_lg

## 1. Data Loading and Preprocessing
We parse the XML file to extract the `Body` (text) and `Tags` (entities) of Questions (`PostTypeId="1"`).

In [ ]:
xml_file = "data/Posts.xml"

def load_data(xml_path, limit=None):
    training_data = []
    context = ET.iterparse(xml_path, events=("end",))
    count = 0
    
    for event, elem in context:
        if elem.tag == "row":
            # process only Questions (PostTypeId=1)
            if elem.get("PostTypeId") == "1":
                body_html = elem.get("Body")
                tags_string = elem.get("Tags")
                
                if body_html and tags_string:
                    # Clean text
                    soup = BeautifulSoup(body_html, "html.parser")
                    text = soup.get_text(separator=" ")
                    
                    # Parse tags: <math><algebra> -> ['math', 'algebra']
                    tags = [t for t in tags_string.replace("<", " ").replace(">", " ").split() if t]
                    
                    # Create entity entries
                    # We look for the tag in the text to create the training example
                    entities = []
                    text_lower = text.lower()
                    
                    for tag in tags:
                        # Simple string matching for this demo
                        # In a real scenario, you might want more robust matching
                        tag_clean = tag.replace("-", " ") # e.g. "linear-algebra" -> "linear algebra"
                        
                        start_index = 0
                        while True:
                            idx = text_lower.find(tag_clean, start_index)
                            if idx == -1:
                                break
                            
                            # Add entity
                            entities.append((idx, idx + len(tag_clean), "TOPIC"))
                            start_index = idx + 1
                    
                    if entities:
                        training_data.append({
                            "text": text,
                            "entities": entities
                        })
                        count += 1
            
            elem.clear()
            if limit and count >= limit:
                break
    return training_data

# Load data (limit to 2000 for quick demonstration, remove limit for full run)
data = load_data(xml_file, limit=2000)
print(f"Loaded {len(data)} examples")
print("Sample:", data[0])

## 2. Prepare Data for Spacy (DocBin)
We convert the raw text and entity offsets into Spacy's binary format.

In [ ]:
nlp = spacy.blank("en")
doc_bin = DocBin()

valid_examples = 0
skipped_examples = 0

for example in tqdm(data):
    text = example['text']
    labels = example['entities']
    doc = nlp.make_doc(text)
    ents = []
    
    for start, end, label in labels:
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is not None:
            ents.append(span)
    
    # Handle overlapping entities (Spacy typically requires non-overlapping)
    filtered_ents = filter_spans(ents)
    doc.ents = filtered_ents
    
    if doc.ents:
        doc_bin.add(doc)
        valid_examples += 1
    else:
        skipped_examples += 1

print(f"processed {valid_examples} valid examples, skipped {skipped_examples} without valid entities")

# Save full dataset first, then split
doc_bin.to_disk("all_data.spacy")

## 3. Train/Dev Split
We split the data manually since we created one DocBin.

In [ ]:
# Reload to split
full_doc_bin = DocBin().from_disk("all_data.spacy")
docs = list(full_doc_bin.get_docs(nlp.vocab))
random.shuffle(docs)

train_size = int(len(docs) * 0.8)
train_docs = docs[:train_size]
dev_docs = docs[train_size:]

train_bin = DocBin(docs=train_docs)
train_bin.to_disk("train.spacy")

dev_bin = DocBin(docs=dev_docs)
dev_bin.to_disk("dev.spacy")

print(f"Training docs: {len(train_docs)}, Dev docs: {len(dev_docs)}")

## 4. Configuration
We create a `base_config.cfg` tailored for NER and then fill it to create the final `config.cfg`.

In [ ]:
base_config_content = """
[paths]
train = null
dev = null
vectors = "en_core_web_lg"
init_tok2vec = null

[system]
gpu_allocator = null
seed = 0

[nlp]
lang = "en"
pipeline = ["tok2vec","ner"]
batch_size = 1000

[components]

[components.tok2vec]
factory = "tok2vec"

[components.ner]
factory = "ner"

[corpora]

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 0
gold_preproc = false
limit = 0
augmenter = null

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 0
gold_preproc = false
limit = 0
augmenter = null

[training]
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"
seed = ${system.seed}
gpu_allocator = ${system.gpu_allocator}
dropout = 0.1
accumulate_gradient = 1
patience = 1600
max_epochs = 0
max_steps = 20000
eval_frequency = 200
frozen_components = []
annotating_components = []
before_to_disk = null

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
discard_oversize = false
tolerance = 0.2
get_length = null

[training.batcher.size]
@schedules = "compounding.v1"
start = 100
stop = 1000
compound = 1.001
t = 0.0

[training.logger]
@loggers = "spacy.ConsoleLogger.v1"
progress_bar = false

[training.optimizer]
@optimizers = "Adam.v1"
beta1 = 0.9
beta2 = 0.999
L2_is_weight_decay = true
L2 = 0.01
grad_clip = 1.0
use_averages = false
eps = 0.00000001
learn_rate = 0.001
"""

with open("base_config.cfg", "w") as f:
    f.write(base_config_content)

In [ ]:
!python -m spacy init fill-config base_config.cfg config.cfg

## 5. Training
Run the training loop.

In [ ]:
!python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy

## 6. Evaluation
Load the best model and test it.

In [ ]:
nlp_best = spacy.load("output/model-best")

test_text = "I am having a discussion about valid math notation in my feature-request."
doc = nlp_best(test_text)

spacy.displacy.render(doc, style="ent", jupyter=True)